# LFM2.5-1.2B Fine-Tuning — Mininio Carb Assistant

Thin Colab wrapper. All logic lives in `finetuning/lfm/train_lfm.py`.

**Prerequisites:**
1. Upload `data/output/lfm/` (train.jsonl + eval.jsonl) as `mininio-data.zip` to the root of your Google Drive
2. The zip contents should be `lfm/train.jsonl` and `lfm/eval.jsonl` (no extra wrapper directory)
3. The notebook clones the repo with the training code

**Runtime:** Colab Free T4 (16GB) — ~40-70 min for 3 epochs
---

In [ ]:
# Install dependencies (Colab-specific pins from Unsloth reference)
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install -q unsloth
else:
    import torch
    v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v == "2.9" else "0.0.32.post2" if v == "2.8" else "0.0.29.post3")
    !pip install -q --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install -q --no-deps unsloth
!pip install -q loguru
print("Dependencies installed.")

In [3]:
# Clone repo & mount Drive, copy data
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/CoGian/mininio-ai-finetuning.git /content/mininio-ai-finetuning 2>/dev/null || (cd /content/mininio-ai-finetuning && git pull)

import os, sys
os.chdir('/content/mininio-ai-finetuning')
sys.path.insert(0, '/content/mininio-ai-finetuning')

# Copy and unzip data
!cp /content/drive/MyDrive/mininio-data.zip /content/ 2>/dev/null
!unzip -o /content/mininio-data.zip -d /content/data/output/ 2>/dev/null
!ls /content/data/output/lfm/ 2>/dev/null || echo "Data not found. Make sure mininio-data.zip is in Drive root."

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
chdir: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
/bin/bash: line 1: cd: /content/mininio-ai-finetuning: No such file or directory


FileNotFoundError: [Errno 2] No such file or directory: '/content/mininio-ai-finetuning'

In [ ]:
# Run training (3 epochs, max_seq_length 4096, QLoRA r16, effective batch 16)
!python -m finetuning.lfm.train_lfm \
    --data-dir /content/data/output \
    --output-dir /content/drive/MyDrive/mininio-checkpoints \
    --epochs 3 \
    --max-seq-length 4096 \
    --batch-size 4 \
    --grad-accum 4 \
    --report-to none

print("\nCheckpoints saved to Drive: /content/drive/MyDrive/mininio-checkpoints/lfm/")

### Smoke Test (optional)
Run a quick inference test to verify the model works.

In [ ]:
# Quick smoke test
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/drive/MyDrive/mininio-checkpoints/lfm/merged_16bit",
    max_seq_length=4096,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

messages = [{"role": "user", "content": "I ate 100g of potatoes. How many carbs?"}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=True,
    return_dict=True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.1,
    top_k=50,
    top_p=0.1,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)